# AI Document Chatbot using RAG

**Module 4 – RAG, Embeddings & Vector Databases**

Pipeline: Document → Text Extraction → Chunking → Embeddings → FAISS Vector Database → Retrieval → LLM → Answer


In [ ]:
!pip install -q pypdf sentence-transformers faiss-cpu transformers torch
print('Installation completed!')


In [ ]:
import os
import numpy as np
import faiss
import torch
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
print('Libraries imported successfully!')


In [ ]:
from google.colab import files
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
print('Uploaded file:', file_name)


In [ ]:
reader = PdfReader(file_name)
text = ''
for page in reader.pages:
    page_text = page.extract_text()
    if page_text:
        text += page_text + '\n'
print('PDF text extracted successfully!')
print('Number of characters:', len(text))
print('\nPreview:\n')
print(text[:3000])


In [ ]:
text = text.replace('\n', ' ')
text = ' '.join(text.split())
print('Text cleaned successfully!')
print(text[:1000])


In [ ]:
chunk_size = 500
overlap = 100
chunks = []
start = 0
while start < len(text):
    end = start + chunk_size
    chunk = text[start:end]
    if chunk.strip():
        chunks.append(chunk)
    start += chunk_size - overlap
print('Total chunks created:', len(chunks))
for i, chunk in enumerate(chunks[:5]):
    print(f'\n--- Chunk {i+1} ---')
    print(chunk)


In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedding_model.encode(chunks, convert_to_numpy=True, show_progress_bar=True)
print('Embeddings created successfully!')
print('Embedding shape:', embeddings.shape)


In [ ]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype('float32'))
print('FAISS vector database created!')
print('Number of vectors:', index.ntotal)


In [ ]:
def retrieve_documents(query, top_k=3):
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding.astype('float32'), top_k)
    results = []
    for i in indices[0]:
        if i < len(chunks):
            results.append(chunks[i])
    return results

print('Retrieval function ready!')


In [ ]:
query = 'What is artificial intelligence?'
results = retrieve_documents(query)
for i, result in enumerate(results):
    print(f'\n--- Retrieved Chunk {i+1} ---')
    print(result)


In [ ]:
model_name = 'google/flan-t5-small'
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForSeq2SeqLM.from_pretrained(model_name)
print('LLM loaded successfully!')


In [ ]:
def generate_answer(question, top_k=3):
    retrieved_chunks = retrieve_documents(question, top_k)
    context = '\n\n'.join(retrieved_chunks)
    prompt = f'''Answer the question using only the information provided in the context.\n\nContext:\n{context}\n\nQuestion:\n{question}\n\nAnswer:'''
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)
    with torch.no_grad():
        outputs = llm.generate(**inputs, max_new_tokens=100)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer, retrieved_chunks

print('RAG answer function ready!')


In [ ]:
question = 'What is artificial intelligence?'
answer, sources = generate_answer(question)
print('Question:')
print(question)
print('\nAnswer:')
print(answer)


In [ ]:
print('Retrieved Context:\n')
for i, source in enumerate(sources):
    print(f'\n--- Source {i+1} ---')
    print(source)


In [ ]:
print('AI Document Chatbot')
print('Ask questions about your uploaded document.')
print("Type 'exit' to stop.\n")
while True:
    question = input('You: ').strip()
    if question.lower() == 'exit':
        print('\nChatbot stopped.')
        break
    if not question:
        print('Please enter a question.\n')
        continue
    try:
        answer, sources = generate_answer(question)
        print('\nAI:', answer)
        print()
    except Exception as e:
        print('\nError:', e)
        print('Please try another question.\n')


# RAG Architecture

Document → Text Extraction → Text Chunking → Embedding Generation → FAISS Vector Database → Semantic Similarity Search → Relevant Context Retrieval → FLAN-T5 LLM → Final Answer

RAG helps ground an LLM's response in retrieved document context and can reduce unsupported answers.
